# 十二种融合模型：独立回归实验

这一节为上面的十二种模型分别增加**连续情感分数回归**实验。模型从 `regression_labels` 读取真实分数，最终头输出 `[B,1]`；预测结果使用 `logits[:,0]`，不做 softmax 或 argmax，也不从分类编号推算连续标签。

**分类与回归分别训练、分别选模、分别保存。**这不是共享骨干的双头多任务训练，也不会把已训练的分类器直接转换成回归器。前面的分类配置、模型对象及结果不被覆盖。每个回归实验的配置由对应分类配置复制而来，保留用户设置的模型结构、预训练、学习率、轮次、随机种子与辅助损失；仅切换 `task="regression"`、关闭类别加权，并在原输出根目录下新增 `regression/`。

主回归损失为 **MAE（L1）**；验证集 MAE 用于 early stopping 和选模。加载验证最佳模型后统一评估一次测试集，报告 MAE、RMSE、Pearson。MAE／RMSE 越低越好，Pearson 越高越好；样本或预测没有方差时 Pearson 为 `None`，不是 0。各模型自己的预训练／辅助目标仍保留，不能把不同模型的训练总损失当作统一回归指标。

可以只运行需要比较的模型小节，然后运行末尾汇总；从上到下全部运行会训练十二个回归模型。这里不自动启动原来的分类实验。若同一 Kernel 中曾导入旧模块，请先重启，再加载数据和原有独立 mask。


In [ ]:
import os
import json
import pickle
from pathlib import Path
import numpy as np

from multi_fusion_model.regression_experiments import (
    make_regression_config,
    get_model_module,
    summarize_regression_experiments,
)

if "data" not in globals():
    # 独立 Notebook 请从项目根目录启动，或先运行自己的数据加载代码。
    regression_data_files = list((Path.cwd() / "datasets").glob("*/aligned_50.pkl"))
    if len(regression_data_files) != 1:
        raise RuntimeError("请先加载 data，或从含 datasets/*/aligned_50.pkl 的项目根目录启动。")
    with regression_data_files[0].open("rb") as regression_file:
        data = pickle.load(regression_file)

if "MODULE_DIR" not in globals():
    MODULE_DIR = str(Path.cwd() / "results" / "second_question")
if "mask_overrides" not in globals():
    mask_overrides = None
if "regression_experiments" not in globals():
    regression_experiments = {}

for regression_split in ("train", "valid", "test"):
    regression_labels = np.asarray(data[regression_split]["regression_labels"])
    if regression_labels.ndim == 2 and regression_labels.shape[1] == 1:
        regression_labels = regression_labels[:, 0]
    if regression_labels.ndim != 1 or len(regression_labels) != len(data[regression_split]["text"]):
        raise ValueError(f"{regression_split}: regression_labels 必须每个样本一个连续分数")
    if not np.isfinite(regression_labels).all():
        raise ValueError(f"{regression_split}: regression_labels 含非有限值")
    print(regression_split, "回归样本数:", len(regression_labels))


## 数据、mask 与结果的可比性

三路输入继续使用原来的 50 个对应槽位；音视频标准化参数只由训练集拟合。`mask_overrides` 原样传递，零特征行不会被自动认定为缺失；若没有显式 mask，仍需满足原有 `text_bert` attention mask 的对齐约定。

以下配置参数从构建时主 Notebook 中各模型**最新的顶层 Config 赋值**复制，不执行原来的训练单元。每段保留独立的 `*_base_config`，经 `make_regression_config` 深复制生成回归配置，修改回归配置不会改动原分类配置或其中的嵌套字典。构建后若另行调整了分类超参数，请同步修改对应回归小节，或重新运行构建脚本。

回归实验不使用 `classification_labels` 训练或选模。M3ER 的乘性分类目标需要单独的回归适配；HyCon 的对比配对、EMOE 的路由监督和 CaReFlow 的样本距离也按连续标签计算，见各小节说明。不同模型仍沿用各自原有实验适配范围，不能仅通过切换任务就视为论文官方基线的严格复现。


## 1. Weighted Sum：回归

加权融合；输出连续分数，不对三个类别编号求平均。

配置来源：原主 Notebook 的 `config`（构建时单元索引 6）。


In [ ]:
import multi_fusion_model.weighted_sum_fusion as weighted_sum_reg_module

weighted_sum_base_config = weighted_sum_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=1,
    ff_dim=256,
    dropout=0.2,
    batch_size=32,
    epochs=40,
    patience=8,
    lr=3e-4,
    weight_decay=1e-4,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    auxiliary_loss_weight=0.0,
    device=None,  # 自动选 CUDA，否则 CPU；也可填 "cpu" 或 "cuda"。
    output_root=str(os.path.join(MODULE_DIR , "weighted_sum_runs")),
)

weighted_sum_reg_config = make_regression_config(weighted_sum_base_config)
weighted_sum_reg_experiment = weighted_sum_reg_module.fit_experiment(
    data, weighted_sum_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["weighted_sum"] = weighted_sum_reg_experiment
print("验证集:", weighted_sum_reg_experiment.valid_metrics)
print("测试集:", weighted_sum_reg_experiment.test_metrics)
print("回归输出目录:", weighted_sum_reg_experiment.run_dir)


## 2. Poria 上下文 LSTM：回归

保留原有 staged／joint 设置；分阶段训练的各个分支也使用回归标签。

配置来源：原主 Notebook 的 `lstm_config`（构建时单元索引 13）。


In [ ]:
import multi_fusion_model.context_lstm_fusion as poria_reg_module

poria_base_config = poria_reg_module.Config(
    task="classification",
    hidden_dim=128,
    context_dim=128,
    fusion_hidden_dim=128,
    fusion_context_dim=128,
    bidirectional=True,
    dropout=0.2,
    pooling="mean",
    training_mode="staged",
    pretrain_epochs=20,
    pretrain_patience=5,
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    seed=42,
     output_root=str(os.path.join(MODULE_DIR , "context_lstm_fusion")),
)

poria_reg_config = make_regression_config(poria_base_config)
poria_reg_experiment = poria_reg_module.fit_experiment(
    data, poria_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["poria"] = poria_reg_experiment
print("验证集:", poria_reg_experiment.valid_metrics)
print("测试集:", poria_reg_experiment.test_metrics)
print("回归输出目录:", poria_reg_experiment.run_dir)


## 3. Ren 2021：回归

交互注意力和序列建模保持原设置，最终头输出一个连续值。

配置来源：原主 Notebook 的 `ren_config`（构建时单元索引 21）。


In [ ]:
import multi_fusion_model.ren2021_fusion as ren2021_reg_module

ren2021_base_config = ren2021_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    dropout=0.2,
    pretrain_epochs=0,
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "ren2021_fusion")),
)

ren2021_reg_config = make_regression_config(ren2021_base_config)
ren2021_reg_experiment = ren2021_reg_module.fit_experiment(
    data, ren2021_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["ren2021"] = ren2021_reg_experiment
print("验证集:", ren2021_reg_experiment.valid_metrics)
print("测试集:", ren2021_reg_experiment.test_metrics)
print("回归输出目录:", ren2021_reg_experiment.run_dir)


## 4. Zheng 2022：回归

保留重构预训练与原来的辅助损失系数，监督阶段改为回归。

配置来源：原主 Notebook 的 `zheng_config`（构建时单元索引 32）。


In [ ]:
import multi_fusion_model.zheng2022_fusion as zheng2022_reg_module

zheng2022_base_config = zheng2022_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    dropout=0.2,
    pretrain_epochs=3,
    pretrain_auxiliary_weights={"reconstruction": 1.0},
    auxiliary_weights={"reconstruction": 0.0},
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "zheng2022_fusion")),
)

zheng2022_reg_config = make_regression_config(zheng2022_base_config)
zheng2022_reg_experiment = zheng2022_reg_module.fit_experiment(
    data, zheng2022_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["zheng2022"] = zheng2022_reg_experiment
print("验证集:", zheng2022_reg_experiment.valid_metrics)
print("测试集:", zheng2022_reg_experiment.test_metrics)
print("回归输出目录:", zheng2022_reg_experiment.run_dir)


## 5. Pan 2020：回归

保留原来的分支预训练及融合训练；两个阶段均按回归目标运行。

配置来源：原主 Notebook 的 `pan_config`（构建时单元索引 40）。


In [ ]:
import multi_fusion_model.pan2020_fusion as pan2020_reg_module

pan2020_base_config = pan2020_reg_module.Config(
    task="classification",
    d_model=128,
    lstm_hidden=128,
    unimodal_layers=2,
    dropout=0.2,
    training_mode="staged",
    pretrain_epochs=20,
    pretrain_patience=5,
    epochs=100,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    branch_loss_weight=1.0,  # 仅 joint 模式使用
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "pan2020_fusion")),
)

pan2020_reg_config = make_regression_config(pan2020_base_config)
pan2020_reg_experiment = pan2020_reg_module.fit_experiment(
    data, pan2020_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["pan2020"] = pan2020_reg_experiment
print("验证集:", pan2020_reg_experiment.valid_metrics)
print("测试集:", pan2020_reg_experiment.test_metrics)
print("回归输出目录:", pan2020_reg_experiment.run_dir)


## 6. M3ER 2020：回归

回归使用连续融合输出与单模态 MAE 辅助项，替代分类的乘性概率目标；这是回归适配。

配置来源：原主 Notebook 的 `m3er_config`（构建时单元索引 45）。


In [ ]:
import multi_fusion_model.m3er2020_fusion as m3er2020_reg_module

m3er2020_base_config = m3er2020_reg_module.Config(
    task="classification",
    d_model=32,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={
        "m3er_memory_dim": 128,
        "m3er_classifier_hidden": 64,
        "m3er_cca_components": 16,
        "m3er_cca_threshold": 0.0,
        "m3er_ridge": 0.01,
        "m3er_fit_max_slots": 8192,
        "m3er_fit_seed": 1729,
        "m3er_beta": 2.0,
        "m3er_proxy_during_training": False,
    },
    auxiliary_weights={
        "m3er_multiplicative": 1.0,
        "m3er_unimodal_regression": 0.1,  # 仅回归任务使用
    },
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "m3er2020_fusion")),
)

m3er2020_reg_config = make_regression_config(m3er2020_base_config)
m3er2020_reg_experiment = m3er2020_reg_module.fit_experiment(
    data, m3er2020_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["m3er2020"] = m3er2020_reg_experiment
print("验证集:", m3er2020_reg_experiment.valid_metrics)
print("测试集:", m3er2020_reg_experiment.test_metrics)
print("回归输出目录:", m3er2020_reg_experiment.run_dir)


## 7. MFRM 2022：回归

多层融合及残差记忆保持不变，训练连续情感分数。

配置来源：原主 Notebook 的 `mfrm_config`（构建时单元索引 54）。


In [ ]:
import multi_fusion_model.mfrm2022_fusion as mfrm2022_reg_module

mfrm2022_base_config = mfrm2022_reg_module.Config(
    task="classification",
    d_model=128,
    num_layers=1,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={"mfrm_intensity_gain": 1.0},
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "mfrm2022_fusion")),
)

mfrm2022_reg_config = make_regression_config(mfrm2022_base_config)
mfrm2022_reg_experiment = mfrm2022_reg_module.fit_experiment(
    data, mfrm2022_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["mfrm2022"] = mfrm2022_reg_experiment
print("验证集:", mfrm2022_reg_experiment.valid_metrics)
print("测试集:", mfrm2022_reg_experiment.test_metrics)
print("回归输出目录:", mfrm2022_reg_experiment.run_dir)


## 8. TransModality 2020：回归

跨模态翻译和循环辅助项保留原系数，最终预测改为回归。

配置来源：原主 Notebook 的 `trans_config`（构建时单元索引 63）。


In [ ]:
import multi_fusion_model.transmodality2020_fusion as transmodality2020_reg_module

transmodality2020_base_config = transmodality2020_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=1,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={"translation_ff_dim": 256},
    auxiliary_weights={
        "translation_text_audio": 0.1,
        "cycle_audio_text": 0.1,
        "translation_text_vision": 0.1,
        "cycle_vision_text": 0.1,
    },
    epochs=60,
    patience=10,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "transmodality2020_fusion")),
)

transmodality2020_reg_config = make_regression_config(transmodality2020_base_config)
transmodality2020_reg_experiment = transmodality2020_reg_module.fit_experiment(
    data, transmodality2020_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["transmodality2020"] = transmodality2020_reg_experiment
print("验证集:", transmodality2020_reg_experiment.valid_metrics)
print("测试集:", transmodality2020_reg_experiment.test_metrics)
print("回归输出目录:", transmodality2020_reg_experiment.run_dir)


## 9. MEmoBERT 2022：回归

保留缓存特征的遮蔽重构预训练设置，微调阶段使用连续分数监督。

配置来源：原主 Notebook 的 `memo_config`（构建时单元索引 72）。


In [ ]:
import multi_fusion_model.memobert2022_fusion as memobert2022_reg_module

memobert2022_base_config = memobert2022_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=2,
    dropout=0.2,
    pretrain_epochs=3,
    model_options={"memobert_mask_probability": 0.15, "memobert_span_length": 3},
    pretrain_auxiliary_weights={
        "memobert_masked_text": 1.0,
        "memobert_masked_audio": 1.0,
        "memobert_masked_vision": 1.0,
    },
    auxiliary_weights={
        "memobert_masked_text": 0.0,
        "memobert_masked_audio": 0.0,
        "memobert_masked_vision": 0.0,
    },
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "memobert2022_fusion")),
)

memobert2022_reg_config = make_regression_config(memobert2022_base_config)
memobert2022_reg_experiment = memobert2022_reg_module.fit_experiment(
    data, memobert2022_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["memobert2022"] = memobert2022_reg_experiment
print("验证集:", memobert2022_reg_experiment.valid_metrics)
print("测试集:", memobert2022_reg_experiment.test_metrics)
print("回归输出目录:", memobert2022_reg_experiment.run_dir)


## 10. HyCon 2022：回归

对比学习的正负配对按情感分数是否 >= 0 分组；这只用于辅助配对，主预测头仍输出连续值，不把回归标签二值化。

配置来源：原主 Notebook 的 `hycon_config`（构建时单元索引 81）。


In [ ]:
import multi_fusion_model.hycon2022_fusion as hycon2022_reg_module

hycon2022_base_config = hycon2022_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=1,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={"hycon_margin": 0.8, "hycon_refinement_weight": 1.0},
    auxiliary_weights={"hycon_intra": 1.0, "hycon_inter": 1.0, "hycon_semi": 1.0},
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "hycon2022_fusion")),
)

hycon2022_reg_config = make_regression_config(hycon2022_base_config)
hycon2022_reg_experiment = hycon2022_reg_module.fit_experiment(
    data, hycon2022_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["hycon2022"] = hycon2022_reg_experiment
print("验证集:", hycon2022_reg_experiment.valid_metrics)
print("测试集:", hycon2022_reg_experiment.test_metrics)
print("回归输出目录:", hycon2022_reg_experiment.run_dir)


## 11. EMOE 2025：回归

主任务和单模态监督用 MAE；路由重要性使用单模态预测的平方误差，在线特征蒸馏保持原设置。

配置来源：原主 Notebook 的 `emoe_config`（构建时单元索引 93）。


In [ ]:
import multi_fusion_model.emoe2025_fusion as emoe2025_reg_module

emoe2025_base_config = emoe2025_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=2,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={
        "emoe_router_hidden": 128,
        "emoe_temperature": 0.1,
        "emoe_reliability_epsilon": 0.1,
    },
    auxiliary_weights={
        "emoe_unimodal": 1.0,
        "emoe_router_fit": 0.01,
        "emoe_router_entropy": 0.1,
        "emoe_distillation": 0.1,
    },
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "emoe2025_fusion")),
)

emoe2025_reg_config = make_regression_config(emoe2025_base_config)
emoe2025_reg_experiment = emoe2025_reg_module.fit_experiment(
    data, emoe2025_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["emoe2025"] = emoe2025_reg_experiment
print("验证集:", emoe2025_reg_experiment.valid_metrics)
print("测试集:", emoe2025_reg_experiment.test_metrics)
print("回归输出目录:", emoe2025_reg_experiment.run_dir)


## 12. CaReFlow 2026：回归

跨样本流匹配的 margin 使用原 margin 加两条样本回归标签的平方差，同样本 margin 仍为 0。

配置来源：原主 Notebook 的 `careflow_config`（构建时单元索引 108）。


In [ ]:
import multi_fusion_model.careflow2026_fusion as careflow2026_reg_module

careflow2026_base_config = careflow2026_reg_module.Config(
    task="classification",
    d_model=128,
    nhead=4,
    num_layers=1,
    dropout=0.2,
    pretrain_epochs=0,
    model_options={
        "careflow_steps": 2,
        "careflow_pair_ratio": 4,
        "careflow_margin": 1e-4,
        "careflow_fusion_hidden": 128,
    },
    auxiliary_weights={
        "careflow_forward_audio": 0.1,
        "careflow_forward_vision": 0.1,
        "careflow_backward_audio": 0.1,
        "careflow_backward_vision": 0.1,
    },
    epochs=40,
    patience=8,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    seed=42,
    standardize_av=True,
    class_weighted_loss=False,
    device=None,
    output_root=str(os.path.join(MODULE_DIR, "careflow2026_fusion")),
)

careflow2026_reg_config = make_regression_config(careflow2026_base_config)
careflow2026_reg_experiment = careflow2026_reg_module.fit_experiment(
    data, careflow2026_reg_config, mask_overrides=mask_overrides,
)
regression_experiments["careflow2026"] = careflow2026_reg_experiment
print("验证集:", careflow2026_reg_experiment.valid_metrics)
print("测试集:", careflow2026_reg_experiment.test_metrics)
print("回归输出目录:", careflow2026_reg_experiment.run_dir)


## 汇总已经完成的回归实验

只汇总 `regression_experiments` 中已经完成的模型；不会重新训练或重新评估测试集。表格保留注册顺序，避免按测试分数自动选择模型。输出 `regression_summary.csv` 和 `regression_summary.json`，包含验证／测试指标及各自 checkpoint 路径。

请使用验证集制定超参数与模型选择规则，测试集只用于最终报告；小样本一轮运行只能验证代码与数据链路，不能用于模型优劣结论。


In [ ]:
if not regression_experiments:
    raise RuntimeError("请先运行至少一个回归模型小节。")
regression_summary = summarize_regression_experiments(
    regression_experiments,
    output_dir=Path(MODULE_DIR) / "regression_comparison",
)
try:
    import pandas as pd
    from IPython.display import display
except ImportError:
    for regression_record in regression_summary:
        print(regression_record)
else:
    display(pd.DataFrame(regression_summary))
print("汇总目录:", Path(MODULE_DIR) / "regression_comparison")


## 单个模型的 MAE 曲线、连续预测与重载

把 `regression_model_key` 改为上方已经训练完成的 key，即可检查该模型。分阶段模型按 stage 分开画验证 MAE，各阶段内的 epoch 单独计数；不要把辅助预训练阶段的 MAE 当作最终 checkpoint 的选择结果。

散点图使用已经保存的测试预测，不再次评估整份测试集。重载检查只取第一个测试样本，去掉两种标签后预测，验证部署推断不需要真实标签。下面的表格展示样本 ID、真实连续分数、预测值与残差（预测减真实）。


In [ ]:
regression_model_key = next(iter(regression_experiments))  # 例如 "careflow2026"
regression_selected = regression_experiments[regression_model_key]
regression_module = get_model_module(regression_selected.config)
regression_saved_prediction = regression_selected.test_predictions
regression_y_true = np.asarray(regression_saved_prediction["targets"]).reshape(-1)
regression_y_pred = np.asarray(regression_saved_prediction["logits"])[:, 0]
regression_prediction_rows = [
    {"id": str(sample_id), "target": float(target), "prediction": float(predicted),
     "residual": float(predicted - target)}
    for sample_id, target, predicted in zip(
        regression_saved_prediction["ids"][:10], regression_y_true[:10], regression_y_pred[:10])
]
for regression_row in regression_prediction_rows:
    print(regression_row)

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("曲线可从各实验 history.json 读取；预测已保存在 test_predictions.npz。")
else:
    regression_fig, regression_axes = plt.subplots(1, 2, figsize=(11, 4))
    regression_history = regression_selected.history
    regression_stages = list(dict.fromkeys(row.get("stage", "train") for row in regression_history))
    for regression_stage in regression_stages:
        regression_records = [row for row in regression_history
                              if row.get("stage", "train") == regression_stage
                              and "mae" in row.get("valid", {})]
        if regression_records:
            regression_axes[0].plot(
                [row["epoch"] for row in regression_records],
                [row["valid"]["mae"] for row in regression_records], label=regression_stage,
            )
    regression_axes[0].set(xlabel="Epoch within stage", ylabel="Validation MAE",
                           title=regression_model_key)
    if regression_axes[0].lines:
        regression_axes[0].legend()
    regression_axes[1].scatter(regression_y_true, regression_y_pred, alpha=0.5, s=15)
    regression_low = float(min(regression_y_true.min(), regression_y_pred.min()))
    regression_high = float(max(regression_y_true.max(), regression_y_pred.max()))
    if regression_low == regression_high:
        regression_low, regression_high = regression_low - 1, regression_high + 1
    regression_axes[1].plot([regression_low, regression_high], [regression_low, regression_high], "k--")
    regression_axes[1].set(xlabel="True sentiment score", ylabel="Predicted sentiment score")
    regression_fig.tight_layout()
    plt.show()

regression_loaded_model, regression_metadata = regression_module.load_experiment_model(
    regression_selected.run_dir / "best.pt",
)
regression_one_sample = {
    key: value[:1] for key, value in data["test"].items()
    if key not in ("classification_labels", "regression_labels")
}
regression_one_sample_masks = {
    modality: value[:1] for modality, value in (mask_overrides or {}).get("test", {}).items()
} or None
regression_reloaded_prediction = regression_module.predict_split(
    regression_loaded_model, regression_one_sample, regression_metadata,
    masks=regression_one_sample_masks,
)
np.testing.assert_allclose(
    regression_reloaded_prediction["logits"][:, 0], regression_y_pred[:1], rtol=1e-4, atol=1e-5,
)
print("重载后连续预测:", regression_reloaded_prediction["logits"][:, 0])


## 回归分数的槽位敏感性

分类解释关注固定类别间的分数差；这里直接解释连续输出：`score_drop = 原始回归预测 − 替换该槽位后的回归预测`。正值表示该次替换使预测降低，负值表示使预测升高；数值单位与回归目标相同。mask 保持不变，替换基线仍是模型输入空间的零值（标准化音视频对应训练均值）。

这衡量缓存特征替换的敏感性，不是可加和的因果贡献。EMOE 会重算路由，M3ER 会重算可靠性／代理分支，CaReFlow 会重新积分；用样本 ID、模态和原槽位索引关联保留的词时间戳、音频区间及视频帧记录。BERT 特征已含上下文，如需判断删除某个原始词的影响，应重新编码原文。


In [ ]:
regression_explanation = regression_module.explain_feature_groups(
    regression_selected, split="test", index=0,
)
print("模型:", regression_model_key, "样本:", regression_explanation["id"])
for regression_row in regression_explanation["rows"][:15]:
    print(regression_row)
